# BraTS 2020 — train, evaluate and diagnose a U-Net on Colab

This notebook:

1. clones the repo and finds the BraTS H5 slices on your Google Drive;
2. checks the mask format (one-hot NCR/ED/ET labels vs nested WT/TC/ET regions);
3. trains with a **patient-level** train / val / test split, picking the checkpoint by val mean Dice;
4. plots the **learning curves**;
5. **tunes the probability threshold on the validation patients** (never on test);
6. evaluates **once on the held-out test patients** (per-patient Dice, IoU, HD95) with bootstrap 95% CIs;
7. plots per-patient distributions, Dice vs tumour size, and best / median / worst cases.

Each plot is followed by a short **reading** cell that turns it into a decision.

Everything that should survive a disconnect (checkpoints, `history.csv`, split file, evaluation CSV, figures)
is written to `RUNS_ROOT` on Drive.

**Runtime:** *Runtime → Change runtime type → GPU* (A100 recommended). Rough time per epoch for the 2D U-Net
with mixed precision: about 2–3 min on an A100 and 10–20 min on a T4. Run with `EPOCHS = 2` first: the log prints
the real time per epoch.

**First session:** the slice files are copied from Drive (slow: ~57k small files) and then saved back to Drive
as a single zip. Later sessions copy and unpack that zip in a few minutes.

In [ ]:
# ---------------- Parameters ----------------
DATASET_DIR = '/content/drive/MyDrive/BraTS2020_Dataset'   # where the Kaggle dataset was downloaded
REPO_URL    = 'https://github.com/alexisvannson/brain-tumor-segmentation.git'
BRANCH      = 'main'

MODEL       = 'unet'     # 2D U-Net. 'unet3d' trains on full 155x240x240 volumes and will not fit
                         # in Colab GPU memory until patch-based training is added.
EPOCHS      = 20
BATCH_SIZE  = 8
LR          = 1e-4
PATIENCE    = 8          # early stopping on val mean Dice
SEED        = 42         # training seed (the patient split has its own fixed seed in the config)
RUN_NAME    = f'{MODEL}_seed{SEED}'

RUNS_ROOT     = '/content/drive/MyDrive/brats_runs'  # persistent outputs
COPY_TO_LOCAL = True     # copy the H5 files to the VM disk: much faster than reading ~57k small files from Drive
MAKE_ZIP_ON_DRIVE = True # after the first copy, save one zip to DATASET_DIR so later sessions load in minutes
AMP           = True     # mixed precision: bf16 on A100 (~2x faster), fp16 + loss scaling on T4
NUM_WORKERS   = None     # data-loading processes; None = number of CPUs (max 8)
USE_WANDB     = False
DISPLAY_CHANNEL = 3      # MRI channel shown in overlays (3 = FLAIR if channels are T1, T1ce, T2, FLAIR)

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime > Change runtime type > GPU')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO_DIR = '/content/brain-tumor-segmentation'
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1
assert os.path.exists('datasets/splits.py'), (
    'This branch has no patient-level split code: push the latest commits or set BRANCH.')
!pip install -q h5py scipy pyyaml tqdm pandas

## Data

The Kaggle export stores one file per axial slice (`volume_{patient}_slice_{n}.h5`, about 57k files). The
cell below finds the folder with the H5 files under `DATASET_DIR` (or unzips an archive if only a `.zip` is
there) and copies it to the VM disk. The copy takes a while once per session but makes every epoch much faster.

In [ ]:
import glob, shutil, subprocess

def find_h5_dir(root):
    best, best_n = None, 0
    for d, _, files in os.walk(root):
        n = sum(f.endswith('.h5') for f in files)
        if n > best_n:
            best, best_n = d, n
    return best, best_n

def run(cmd):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

LOCAL_ROOT = '/content/brats_local'
ZIP_NAME   = 'brats_h5_slices.zip'          # written by this notebook on the first session
own_zip    = os.path.join(DATASET_DIR, ZIP_NAME)

local_dir, n_local = find_h5_dir(LOCAL_ROOT) if os.path.isdir(LOCAL_ROOT) else (None, 0)
if n_local:                                  # already prepared earlier in this session
    DATA_DIR = local_dir
else:
    zips = [own_zip] if os.path.exists(own_zip) else \
           glob.glob(os.path.join(DATASET_DIR, '**', '*.zip'), recursive=True)
    if zips:
        # One big file copies from Drive far faster than thousands of small ones
        print(f'Unpacking {zips[0]} ({os.path.getsize(zips[0]) / 1e9:.1f} GB) ...')
        os.makedirs(LOCAL_ROOT, exist_ok=True)
        run(f'cp "{zips[0]}" /content/_data.zip')
        run(f'unzip -q -o /content/_data.zip -d {LOCAL_ROOT} && rm /content/_data.zip')
        DATA_DIR, n_local = find_h5_dir(LOCAL_ROOT)
        if not n_local:
            raise FileNotFoundError(f'{zips[0]} contains no .h5 files')
    else:
        h5_dir, n_h5 = find_h5_dir(DATASET_DIR)
        if not n_h5:
            raise FileNotFoundError(f'No .h5 files or .zip archive under {DATASET_DIR}')
        DATA_DIR = h5_dir
        if COPY_TO_LOCAL:
            sample = [os.path.join(h5_dir, f) for f in sorted(os.listdir(h5_dir)) if f.endswith('.h5')][:50]
            est = sum(os.path.getsize(f) for f in sample) / len(sample) * n_h5
            free = shutil.disk_usage('/content').free
            print(f'Dataset ~{est / 1e9:.1f} GB, free VM disk {free / 1e9:.1f} GB')
            if est > 0.9 * free:
                print('Not enough local disk: reading directly from Drive (slower).')
            else:
                print(f'Copying {n_h5} files to the VM (slow the first time, many small files) ...')
                DATA_DIR = os.path.join(LOCAL_ROOT, 'data')
                os.makedirs(DATA_DIR, exist_ok=True)
                run(f'cp -r "{h5_dir}/." "{DATA_DIR}/"')
                if MAKE_ZIP_ON_DRIVE and est * 1.6 < free:
                    print('Saving a zip to Drive for faster loading next time ...')
                    run(f'cd {LOCAL_ROOT} && zip -q -1 -r /content/{ZIP_NAME} data')
                    run(f'cp /content/{ZIP_NAME} "{own_zip}" && rm /content/{ZIP_NAME}')
                    print(f'Saved {own_zip}')

n_patients = len({f.split('_slice_')[0] for f in os.listdir(DATA_DIR) if f.endswith('.h5')})
n_files = sum(f.endswith('.h5') for f in os.listdir(DATA_DIR))
print(f'DATA_DIR = {DATA_DIR}  ({n_files} slice files, {n_patients} patients)')

In [ ]:
# One-hot labels (NCR, ED, ET) never overlap; nested regions (WT ⊇ TC ⊇ ET) do.
out = subprocess.run(['python', 'scripts/check_masks.py', DATA_DIR, '--n', '1000'],
                     capture_output=True, text=True, check=True).stdout
print(out)
if 'mask_format: regions' in out:
    MASK_FORMAT = 'regions'
elif 'mask_format: labels' in out:
    MASK_FORMAT = 'labels'
else:
    raise RuntimeError('Could not determine the mask format (no foreground sampled?). Re-run with a larger --n.')
print('Using mask_format =', MASK_FORMAT)

In [ ]:
import yaml
RUN_DIR = os.path.join(RUNS_ROOT, RUN_NAME)
FIG_DIR = os.path.join(RUN_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

cfg = yaml.safe_load(open(f'configs/{MODEL}.yaml'))
cfg['data'].update(
    data_dir=DATA_DIR,
    # One split file shared by every run, so all models are compared on the same test patients
    split_file=os.path.join(RUNS_ROOT, f"split_seed{cfg['data'].get('split_seed', 42)}.json"),
    mask_format=MASK_FORMAT,
)
NUM_WORKERS = NUM_WORKERS or min(8, os.cpu_count() or 2)
cfg['training'].update(seed=SEED, checkpoint_dir=RUN_DIR, epochs=EPOCHS, batch_size=BATCH_SIZE,
                       lr=LR, patience=PATIENCE, num_workers=NUM_WORKERS, amp=AMP)
print(f'{os.cpu_count()} CPUs -> num_workers = {NUM_WORKERS} | AMP = {AMP}')
CONFIG_PATH = os.path.join(RUN_DIR, 'run_config.yaml')
with open(CONFIG_PATH, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open(CONFIG_PATH).read())

## Training

Checkpoints and logs go to `RUN_DIR` on Drive: `best.pth` (highest val mean Dice), `{MODEL}.pth` (last
epoch), `history.csv` and `training_logs_*.txt`.

If the session disconnects, the plots below still work from the `history.csv` on Drive. To continue training,
change `RUN_NAME` (otherwise `history.csv` is overwritten) and add `--weights {RUN_DIR}/best.pth` to the
command.

In [ ]:
os.environ['WANDB_MODE'] = 'online' if USE_WANDB else 'disabled'
os.environ['TQDM_MININTERVAL'] = '15'   # fewer progress-bar refreshes in the notebook output
!python scripts/training.py {MODEL} --config {CONFIG_PATH}

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, NullFormatter

REGIONS = ['whole_tumor', 'tumor_core', 'enhancing_tumor']
SHORT   = {'whole_tumor': 'WT', 'tumor_core': 'TC', 'enhancing_tumor': 'ET'}
LABEL   = {'whole_tumor': 'Whole tumor', 'tumor_core': 'Tumor core', 'enhancing_tumor': 'Enhancing tumor'}
COLOR   = {'whole_tumor': '#2a78d6', 'tumor_core': '#eb6834', 'enhancing_tumor': '#1baf7a'}
TRAIN_C, VAL_C = '#2a78d6', '#eb6834'
INK, MUTED, GRID, SURFACE = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'axes.edgecolor': '#b5b4ae', 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titlelocation': 'left', 'axes.titlecolor': INK,
    'axes.labelcolor': MUTED, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'lines.linewidth': 2, 'legend.frameon': False, 'font.size': 10,
})

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name), dpi=150, bbox_inches='tight')

## 1 · Learning curves

**Loss** shows over- or underfitting, **val Dice per region** shows when each region stops improving (enhancing
tumour usually lags), and **learning rate** shows when ReduceLROnPlateau kicked in.

In [ ]:
hist = pd.read_csv(os.path.join(RUN_DIR, 'history.csv'))
best_i = hist['val_mean_dice'].idxmax()
best_epoch = int(hist.loc[best_i, 'epoch'])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
ax = axes[0]
ax.plot(hist.epoch, hist.train_loss, color=TRAIN_C, label='Train')
ax.plot(hist.epoch, hist.val_loss, color=VAL_C, label='Validation')
ax.set(title='Loss (BCE + soft Dice)', xlabel='Epoch', ylabel='Loss')
ax.legend()

ax = axes[1]
for r in REGIONS:
    ax.plot(hist.epoch, hist[f'val_dice_{r}'], color=COLOR[r], label=LABEL[r])
ax.plot(hist.epoch, hist.val_mean_dice, color=MUTED, linestyle='--', linewidth=1.5, label='Mean')
ax.axvline(best_epoch, color=MUTED, linewidth=1, linestyle=':')
ax.annotate(f'best: epoch {best_epoch}\nmean Dice {hist.val_mean_dice[best_i]:.3f}',
            xy=(best_epoch, hist.val_mean_dice[best_i]), xytext=(6, -28), textcoords='offset points',
            color=INK, fontsize=9)
ax.set(title='Validation Dice per patient (threshold 0.5)', xlabel='Epoch', ylabel='Dice', ylim=(0, 1))
ax.legend(loc='lower right')

ax = axes[2]
ax.plot(hist.epoch, hist.lr, color=TRAIN_C)
ax.set_yscale('log')
ax.set(title='Learning rate', xlabel='Epoch', ylabel='LR')
fig.tight_layout()
save(fig, '1_learning_curves.png')
plt.show()

In [ ]:
# Turn the curves into decisions
last = int(hist.epoch.iloc[-1])
gap = hist.val_loss.iloc[-1] - hist.train_loss.iloc[-1]
val_loss_min_epoch = int(hist.loc[hist.val_loss.idxmin(), 'epoch'])
n_lr_drops = int((hist.lr.diff() < 0).sum())
print(f'Best epoch {best_epoch}/{last} | val-train loss gap at end: {gap:+.3f} | LR reductions: {n_lr_drops}')
if best_epoch >= last - 1 and last >= EPOCHS:
    print('- Val Dice was still improving at the last epoch: training is budget-limited. Increase EPOCHS '
          '(or resume from best.pth) before tuning anything else.')
if val_loss_min_epoch < last - 3 and hist.train_loss.iloc[-1] < hist.train_loss[hist.epoch == val_loss_min_epoch].item():
    print(f'- Val loss bottomed at epoch {val_loss_min_epoch} while train loss kept falling: overfitting. '
          'Candidates: augmentation, weight decay, InstanceNorm + dropout.')
weakest = min(REGIONS, key=lambda r: hist.loc[best_i, f'val_dice_{r}'])
print(f'- Weakest region at the best epoch: {LABEL[weakest]} '
      f'({hist.loc[best_i, f"val_dice_{weakest}"]:.3f}). See the size analysis below.')

## 2 · Threshold tuning on the validation patients

The sigmoid threshold 0.5 is arbitrary. Here every val patient is predicted once and Dice is computed for a
grid of thresholds (shaded bands = 95% CI of the mean across patients). A new threshold is adopted only if it
improves val mean Dice by more than 0.005; smaller gains are within noise and would just overfit the val set.

In [ ]:
import sys
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
sys.path.insert(0, os.getcwd())
from scripts.training import create_model, get_split, volume_dataset, predict
from scripts.metrics import dice_iou_per_case

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BEST_WEIGHTS = os.path.join(RUN_DIR, 'best.pth')
model = create_model(MODEL, cfg).to(device)
model.load_state_dict(torch.load(BEST_WEIGHTS, map_location=device))
model.eval()
split = get_split(cfg)
SLICE_BATCH = 2 * BATCH_SIZE

thresholds = np.round(np.arange(0.1, 0.91, 0.05), 2)
val_ds = volume_dataset(cfg, MODEL, split['val'])
dice_t = np.zeros((len(val_ds), len(thresholds), 3))
with torch.no_grad():
    for i, (img, mask) in enumerate(tqdm(DataLoader(val_ds, batch_size=1, num_workers=NUM_WORKERS), desc='val patients')):
        prob = torch.sigmoid(predict(model, img.to(device), SLICE_BATCH))
        tgt = mask.to(device) > 0.5
        for j, t in enumerate(thresholds):
            dice_t[i, j] = dice_iou_per_case(prob > t, tgt)[0][0].cpu().numpy()

mean_t = dice_t.mean(axis=0)                                   # (thresholds, regions)
ci_t = 1.96 * dice_t.std(axis=0, ddof=1) / np.sqrt(len(val_ds))
overall = mean_t.mean(axis=1)
j05, jbest = int(np.argmin(np.abs(thresholds - 0.5))), int(overall.argmax())
THRESHOLD = float(thresholds[jbest]) if overall[jbest] - overall[j05] > 0.005 else 0.5

fig, ax = plt.subplots(figsize=(8, 4.5))
for k, r in enumerate(REGIONS):
    ax.plot(thresholds, mean_t[:, k], color=COLOR[r], label=LABEL[r])
    ax.fill_between(thresholds, mean_t[:, k] - ci_t[:, k], mean_t[:, k] + ci_t[:, k], color=COLOR[r], alpha=0.15, linewidth=0)
ax.plot(thresholds, overall, color=MUTED, linestyle='--', linewidth=1.5, label='Mean')
ax.axvline(THRESHOLD, color=INK, linewidth=1, linestyle=':')
ax.annotate(f'chosen {THRESHOLD:.2f}', xy=(THRESHOLD, 1), xycoords=('data', 'axes fraction'),
            xytext=(4, -14), textcoords='offset points', color=INK, fontsize=9)
ax.set(title=f'Validation Dice vs threshold (n={len(val_ds)} patients)', xlabel='Sigmoid threshold', ylabel='Dice')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=4)
fig.tight_layout()
save(fig, '2_threshold_sweep.png')
plt.show()

print(f'Mean val Dice at 0.5: {overall[j05]:.4f} | best {overall[jbest]:.4f} at {thresholds[jbest]:.2f} '
      f'-> using THRESHOLD = {THRESHOLD}')
if jbest in (0, len(thresholds) - 1):
    print(f'  Note: the optimum is at the edge of the grid ({thresholds[jbest]:.2f}). This usually means the model is '
          'under-trained or badly calibrated; train longer before reading much into the threshold.')
for k, r in enumerate(REGIONS):
    print(f'  {LABEL[r]:<16} best threshold {thresholds[mean_t[:, k].argmax()]:.2f} '
          f'(Dice {mean_t[:, k].max():.4f} vs {mean_t[j05, k]:.4f} at 0.5)')

## 3 · Final evaluation on the held-out test patients

Run **once**, with the checkpoint and threshold chosen on validation data. Re-tuning anything after looking at
these numbers would turn the test set into a second validation set.

In [ ]:
EVAL_TXT = os.path.join(RUN_DIR, 'eval', 'test.txt')
!python scripts/evaluate.py {MODEL} {BEST_WEIGHTS} --config {CONFIG_PATH} --split test --threshold {THRESHOLD} --slice_batch {SLICE_BATCH} --output {EVAL_TXT}
res = pd.read_csv(os.path.splitext(EVAL_TXT)[0] + '.csv', dtype={'case': str})

In [ ]:
rng = np.random.default_rng(0)
def boot_ci(x, n=10000):
    x = np.asarray(x, dtype=float)
    means = x[rng.integers(0, len(x), (n, len(x)))].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

rows = []
for metric in ['dice', 'iou', 'hd95']:
    for r in REGIONS:
        v = res[f'{metric}_{r}']
        lo, hi = boot_ci(v)
        rows.append({'metric': metric, 'region': LABEL[r], 'mean': v.mean(), '95% CI low': lo, '95% CI high': hi,
                     'median': v.median(), 'std': v.std()})
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(RUN_DIR, 'eval', 'test_summary_ci.csv'), index=False)
print(f'Test set: {len(res)} patients, threshold {THRESHOLD}')
summary.round(4)

## 4 · Per-patient distributions

A mean hides failures. Each dot is one test patient. Dice = 0 with HD95 = 373 means the region was entirely
missed, or predicted in a patient who doesn't have it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, metric, title, ylab in [(axes[0], 'dice', 'Dice per test patient', 'Dice'),
                                (axes[1], 'hd95', 'HD95 per test patient', 'HD95 (mm)')]:
    data = [res[f'{metric}_{r}'].values for r in REGIONS]
    bp = ax.boxplot(data, widths=0.45, showfliers=False, patch_artist=True,
                    medianprops=dict(color=INK, linewidth=1.5),
                    boxprops=dict(facecolor='none', edgecolor=MUTED), whiskerprops=dict(color=MUTED),
                    capprops=dict(color=MUTED))
    for k, (r, v) in enumerate(zip(REGIONS, data)):
        x = k + 1 + rng.uniform(-0.15, 0.15, len(v))
        ax.scatter(x, v, s=22, color=COLOR[r], alpha=0.75, edgecolor=SURFACE, linewidth=0.8, zorder=3)
    stat = (lambda v: f'mean {v.mean():.3f}') if metric == 'dice' else (lambda v: f'median {np.median(v):.1f}')
    ax.set_xticks(range(1, 4), [f'{LABEL[r]}\n{stat(v)}' for r, v in zip(REGIONS, data)])
    ax.set(title=title, ylabel=ylab)
    if metric == 'hd95' and max(v.max() for v in data) > 50:   # missed regions sit at 373: compress the top
        ax.set_yscale('symlog', linthresh=20)
        ticks = [0, 5, 10, 20, 50, 100, 200, 373]
        ax.set_yticks(ticks, [str(t) for t in ticks])
        ax.yaxis.set_minor_formatter(NullFormatter())
    if metric == 'hd95':
        ax.set_ylim(bottom=0)
axes[0].set_ylim(-0.03, 1.03)
fig.tight_layout()
save(fig, '3_test_distributions.png')
plt.show()

missed = {LABEL[r]: int((res[f'dice_{r}'] == 0).sum()) for r in REGIONS}
print('Patients with Dice = 0 per region:', missed)

## 5 · Dice vs tumour size

Is the model failing on small tumours? Each dot is a test patient. The x-axis is the ground-truth region
volume (1 voxel = 1 mm³). Patients without that region are listed in the panel title, since a log axis can't
place them.

In [ ]:
test_ds = volume_dataset(cfg, MODEL, split['test'])
vols = []
for pid, (img, mask) in zip(test_ds.patient_ids, tqdm(test_ds, desc='test volumes')):
    vols.append({'case': pid, **{f'vol_{r}': mask[k].sum().item() / 1000 for k, r in enumerate(REGIONS)}})
res = res.merge(pd.DataFrame(vols), on='case')

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharey=True)
for ax, r in zip(axes, REGIONS):
    present = res[res[f'vol_{r}'] > 0]
    absent = res[res[f'vol_{r}'] == 0]
    ax.scatter(present[f'vol_{r}'], present[f'dice_{r}'], s=26, color=COLOR[r], alpha=0.8,
               edgecolor=SURFACE, linewidth=0.8)
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    ax.xaxis.set_minor_formatter(NullFormatter())
    if len(present) >= 5:
        ax.annotate(f"Spearman ρ = {present[f'vol_{r}'].corr(present[f'dice_{r}'], method='spearman'):.2f}",
                    xy=(0.03, 0.05), xycoords='axes fraction', color=INK, fontsize=9)
    note = f' · {len(absent)} without it (Dice {absent[f"dice_{r}"].mean():.2f})' if len(absent) else ''
    ax.set(title=f'{LABEL[r]}{note}', xlabel='Ground-truth volume (cm³)', ylim=(-0.03, 1.03))
axes[0].set_ylabel('Dice')
fig.tight_layout()
save(fig, '4_dice_vs_volume.png')
plt.show()
res.to_csv(os.path.splitext(EVAL_TXT)[0] + '_with_volumes.csv', index=False)

## 6 · Best, median and worst test patients

Rows are ranked by mean Dice over the three regions. Each row shows the axial slice with the largest
ground-truth tumour area. Contours: whole tumour, tumour core, enhancing tumour.

In [ ]:
res['mean_dice'] = res[[f'dice_{r}' for r in REGIONS]].mean(axis=1)
ranked = res.sort_values('mean_dice', ascending=False).reset_index(drop=True)
picks = [('Best', ranked.iloc[0]), ('Median', ranked.iloc[len(ranked) // 2]), ('Worst', ranked.iloc[-1])]

fig, axes = plt.subplots(len(picks), 3, figsize=(12, 4 * len(picks)))
for row, (tag, rec) in zip(axes, picks):
    img, mask = test_ds[test_ds.patient_ids.index(rec['case'])]
    with torch.no_grad():
        pred = (torch.sigmoid(predict(model, img[None].to(device), SLICE_BATCH)) > THRESHOLD)[0].cpu()
    z = int(mask[0].sum(dim=(1, 2)).argmax())
    mri = img[DISPLAY_CHANNEL, z].numpy()
    brain = mri[mri != 0]
    vmin, vmax = (np.percentile(brain, [1, 99]) if brain.size else (0, 1))
    for ax, title, seg in [(row[0], 'MRI', None), (row[1], 'Ground truth', mask), (row[2], 'Prediction', pred)]:
        ax.imshow(mri, cmap='gray', vmin=vmin, vmax=vmax)
        if seg is not None:
            for k, r in enumerate(REGIONS):
                if seg[k, z].any():
                    ax.contour(seg[k, z].float().numpy(), levels=[0.5], colors=[COLOR[r]], linewidths=1.5)
        ax.set_title(title, color=INK)
        ax.axis('off')
    dice_txt = '  '.join(f"{SHORT[r]} {rec[f'dice_{r}']:.2f}" for r in REGIONS)
    row[0].set_title(f"{tag} · patient {rec['case']} · slice {z}\nDice  {dice_txt}", color=INK, loc='left')
handles = [Line2D([], [], color=COLOR[r], linewidth=2, label=f'{LABEL[r]} ({SHORT[r]})') for r in REGIONS]
fig.legend(handles=handles, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=(0, 0.03, 1, 1))
save(fig, '5_best_median_worst.png')
plt.show()

## Outputs

Everything is in `RUN_DIR` on Drive:

| File | Content |
|---|---|
| `best.pth`, `{MODEL}.pth` | best-val-Dice and last-epoch weights |
| `history.csv`, `training_logs_*.txt` | per-epoch losses, per-region val Dice, LR |
| `run_config.yaml` | exact config used |
| `eval/test.txt`, `eval/test.csv` | test summary and per-patient Dice / IoU / HD95 |
| `eval/test_summary_ci.csv` | means with bootstrap 95% CIs |
| `eval/test_with_volumes.csv` | per-patient metrics + ground-truth region volumes |
| `figures/*.png` | all plots above |

The patient split is in `RUNS_ROOT/split_seed42.json` and is shared by every run. Copy it into the repo's
`splits/` folder and commit it so the test set is fixed for good.

**Comparing two runs** (different seed, loss or architecture): use the per-patient CSVs with a paired
Wilcoxon signed-rank test (`scipy.stats.wilcoxon(a['dice_enhancing_tumor'], b['dice_enhancing_tumor'])`),
not just the difference of means.